In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────────────
# Run once. You can install via requirements.txt:
# %pip install -q -r requirements.txt

# Or run the direct installs below:
%pip install -q \
    langchain langchain-community langchain-core langchain-text-splitters \
    langchain-chroma langchain-huggingface \
    chromadb \
    pypdf python-docx beautifulsoup4 requests \
    tiktoken \
    langchain-openai langchain-anthropic langchain-ollama langchain-google-genai \
    sentence-transformers python-dotenv

print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
# Edit this cell to switch providers. Everything else adapts automatically.

from rag import RAGConfig

# ── Provider & Model Selection ───────────────────────────────────────────────
config = RAGConfig(
    llm_provider="ollama",            # "openai" | "anthropic" | "ollama" | "google"
    embedding_provider="huggingface", # "openai" | "huggingface" | "ollama" | "google"

    # Optional custom models (leave empty to use provider defaults)
    llm_model="",
    embedding_model="",

    # ChromaDB & Chunking Settings
    chroma_persist_dir="./chroma_db",
    collection_name="rag_docs",
    chunk_size=1000,
    chunk_overlap=150,
    top_k=5,
    search_type="mmr",  # "mmr" | "similarity"
)

print(f"LLM:        {config.llm_provider} / {config.llm_model}")
print(f"Embeddings: {config.embedding_provider} / {config.embedding_model}")
print(f"Vector DB:  ChromaDB → {config.chroma_persist_dir}/{config.collection_name}")
print(f"Search:     {config.search_type}, top_k={config.top_k}")

In [ ]:
# ── Cell 3: Load & Ingest Documents ──────────────────────────────────────────
from rag import RAGPipeline
from langchain_core.documents import Document

# ✏️  Edit the lists below to point at YOUR data sources.
# Leave any list empty [] if you don't have that type.

PDF_PATHS  = [
    # "./docs/report.pdf",
]
DOCX_PATHS = [
    # "./docs/notes.docx",
]
TEXT_PATHS = [
    # "./docs/readme.md",
]
WEB_URLS   = [
    # "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
]

# ── Initialize Pipeline ───────────────────────────────────────────────────────
pipeline = RAGPipeline(config)

if any([PDF_PATHS, DOCX_PATHS, TEXT_PATHS, WEB_URLS]):
    chunks = pipeline.ingest(
        pdf_paths=PDF_PATHS,
        docx_paths=DOCX_PATHS,
        text_paths=TEXT_PATHS,
        web_urls=WEB_URLS,
    )
    print(f"\n✅ Ingested {len(chunks)} chunks into ChromaDB.")
else:
    count = pipeline.vectorstore._collection.count()
    if count > 0:
        print(f"ℹ️  Using existing index with {count} chunks (no new documents to add).")
    else:
        # No documents configured and index is empty — load sample docs for demo
        from rag.ingest import chunk_documents
        sample_docs = [
            Document(
                page_content="Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with generation. It retrieves relevant documents from a knowledge base and uses them as context when generating responses.",
                metadata={"source": "sample_rag", "type": "concept"},
            ),
            Document(
                page_content="The RAG architecture consists of three main components: a retriever, a vector store, and a generator. The retriever fetches relevant documents, which are then provided as context to the generator.",
                metadata={"source": "sample_rag", "type": "architecture"},
            ),
            Document(
                page_content="Vector embeddings are numerical representations of text that capture semantic meaning. They allow similarity search in high-dimensional space, enabling the retriever to find relevant documents based on user queries.",
                metadata={"source": "sample_rag", "type": "embeddings"},
            ),
            Document(
                page_content="ChromaDB is a popular vector database for storing embeddings and performing similarity search. It provides a simple interface for document storage and retrieval.",
                metadata={"source": "sample_rag", "type": "tools"},
            ),
        ]
        chunks = chunk_documents(
            sample_docs,
            chunk_size=config.chunk_size,
            chunk_overlap=config.chunk_overlap,
        )
        pipeline.vectorstore.add_documents(chunks)
        print(f"✅ Loaded {len(chunks)} sample chunks into ChromaDB.")

In [ ]:
# ── Cell 4: Query the Pipeline ────────────────────────────────────────────────
# Replace with questions relevant to YOUR documents.

answer = pipeline.ask("What is RAG and how does it work?", show_sources=True)

In [ ]:
# ── Cell 5: Batch Evaluation ─────────────────────────────────────────────────
# Define question + expected keywords to do a quick sanity check.

eval_cases = [
    # {"question": "What is the main topic?",   "must_contain": ["topic_keyword"]},
    # {"question": "Who are the authors?",       "must_contain": ["author_name"]},
]

pipeline.evaluate(eval_cases)

---

## Tuning Guide

| Problem | What to try |
|---|---|
| Answers miss key facts | Increase `TOP_K` or reduce `CHUNK_SIZE` so chunks are more focused |
| Answers are too fragmented | Increase `CHUNK_SIZE` and `CHUNK_OVERLAP` |
| LLM hallucinates | Lower temperature (already 0); tighten the prompt |
| Slow embedding | Switch `EMBEDDING_PROVIDER` to `"huggingface"` (local, no API cost) |
| Retrieval returns irrelevant chunks | Switch `search_type` from `"mmr"` to `"similarity"` |
| Too expensive | Use `gpt-4o-mini` or switch to `ollama` for fully local inference |

## Next steps

- **Hybrid search** — combine BM25 keyword search with dense vector retrieval (`EnsembleRetriever`)
- **Re-ranking** — add a cross-encoder reranker (e.g. `FlashrankRerank`) after initial retrieval  
- **Metadata filtering** — filter by document type, date, or author at query time  
- **Multi-query retrieval** — generate 3–5 query variants and union the results  
- **Evaluation framework** — use RAGAS or LangChain Evals for faithfulness/relevance scoring